In [1]:
import cv2
import json
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt


from pathlib import Path
from natsort import natsorted
from tqdm import tqdm
import os
from glob import glob

from Config import COLOR_DICT
from scipy.special import softmax
from Exporter import PageXMLExporter
from Utils import unpatch_prediction, pad_image, patch_image, prepare_img_patches, create_dir, resize_to_width, Bbox, LineData, LayoutData, rotate_from_hough

print(ort.__version__)
print(ort.get_available_providers())


1.16.0
['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
def show_image(image: np.array, cmap: str = "") -> None:
    plt.figure(figsize=(18, 9))
    #plt.axis("off")
    
    if cmap != "":
        plt.imshow(image, cmap=cmap)
    else:
        plt.imshow(image)

In [3]:
layout_config = "D:/Data/LayoutAnalysis/Perig/512_v2/2024-1-4_17-17/model_config.json"
patch_size = 512
_file = open(layout_config)
json_content = json.loads(_file.read())
onnx_model_file = json_content["model"]
print(onnx_model_file)
input_width = json_content["input_width"]
input_height = json_content["input_height"]
class_dict = json_content["classes"]
model_file_path = f"D:/Data/LayoutAnalysis/Perig/512_v2/2024-1-4_17-17/{onnx_model_file}"
#model_file_path = f"D:/Datasets/Tibetan/LayoutAnalysis/TrainingSets/january_v1/512/2024-1-9_12-44/{onnx_model_file}"
#execution_providers = ["CPUExecutionProvider", "CUDAExecutionProvider"]
execution_providers = ["CUDAExecutionProvider"]
layout_inference = ort.InferenceSession(model_file_path, providers=execution_providers)

segmentation_model_gpu.onnx


In [4]:
def predict_image(image: np.array):
    padded_img, (pad_x, pad_y) = pad_image(image, patch_size, pad_value=0)
    img_patches, y_steps = patch_image(padded_img, patch_size=patch_size)

    img_batch = [prepare_img_patches(x) for x in img_patches]
    img_batch = np.array(img_batch)
    img_batch = np.transpose(img_batch, axes=[0, 3, 1, 2])
    #img_batch = np.expand_dims(img_batch, axis=1)
    #print(f"Input: {img_batch.shape}")
    img_bach = ort.OrtValue.ortvalue_from_numpy(img_batch)
    pred_batch = layout_inference.run(None, {"input": img_batch})
    pred_batch = pred_batch[0]
    pred_batch = np.transpose(pred_batch, axes=[0, 2, 3, 1])
    pred_batch = softmax(pred_batch, axis=-1)

    unpatched_predition = unpatch_prediction(pred_batch, y_splits=y_steps)
    cropped_image = unpatched_predition[: unpatched_predition.shape[0] - pad_y, : unpatched_predition.shape[1] - pad_x]
    # back_sized = cv2.resize(cropped_image, (int(cropped_image.shape[1] / resize_factor), int(cropped_image.shape[0] / resize_factor)))
    back_sized_image = cv2.resize(cropped_image, (image.shape[1], image.shape[0]))

    combined_output = np.argmax(back_sized_image, axis=-1)
    combined_output = combined_output.astype(np.uint8)
    combined_output = np.expand_dims(combined_output, axis=-1)

    return back_sized_image, combined_output

In [5]:
def generate_preview(
    image: np.array, layout_data: LayoutData, alpha: float = 0.4
) -> np.array:
    preview_image = image.copy()
    preview_mask = np.zeros(image.shape, dtype=np.uint8)

    if len(layout_data.lines) > 0:
        color = tuple([int(x) for x in COLOR_DICT["line"].split(",")])

        for line_data in layout_data.lines:
            cv2.drawContours(
                preview_mask,
                [line_data.contour],
                contourIdx=-1,
                color=color,
                thickness=-1,
            )

    if len(layout_data.images) > 0:
        color = tuple([int(x) for x in COLOR_DICT["image"].split(",")])

        for bbox in layout_data.images:
            cv2.rectangle(
                preview_mask,
                (bbox.x, bbox.y),
                (bbox.x + bbox.w, bbox.y + bbox.h),
                color,
                2,
            )

    if len(layout_data.captions) > 0:
        color = tuple([int(x) for x in COLOR_DICT["caption"].split(",")])

        for bbox in layout_data.captions:
            cv2.rectangle(
                preview_mask,
                (bbox.x, bbox.y),
                (bbox.x + bbox.w, bbox.y + bbox.h),
                color,
                2,
            )

    if len(layout_data.margins) > 0:
        color = tuple([int(x) for x in COLOR_DICT["margin"].split(",")])

        for bbox in layout_data.margins:
            cv2.rectangle(
                preview_mask,
                (bbox.x, bbox.y),
                (bbox.x + bbox.w, bbox.y + bbox.h),
                color,
                2,
            )

    cv2.addWeighted(preview_mask, alpha, preview_image, 1 - alpha, 0, preview_image)

    return preview_image



def generate_page_data(predictions: np.array, area_threshold: int = 20) -> np.array:
        
    # TODO: move the calculation of the text bounding box after the filtering of the lines so that small outlier contours don't extend the box too much
    # TODO: add filtering for the other classes
    # TODO: maybe use cv2.minAreaRect instead of bounding rect?


        pred_images = predictions[:, :, 1]
        pred_lines = predictions[:, :, 2]
        pred_margin = predictions[:, :, 3]
        pred_caption = predictions[:, :, 4]

        pred_images = np.where(pred_images > 200, 255, 0)
        pred_lines = np.where(pred_lines > 200, 255, 0)
        pred_margin = np.where(pred_margin > 200, 255, 0)
        pred_caption = np.where(pred_caption > 200, 255, 0)

        pred_images = pred_images.astype(np.uint8)
        pred_lines = pred_lines.astype(np.uint8)
        pred_margin = pred_margin.astype(np.uint8)
        pred_caption = pred_caption.astype(np.uint8)


        images = []
        text_bboxes = []
        lines = []
        captions = []
        margins = []

        prediction_images = {}
        

        if np.sum(pred_lines) > 0:
            
            
            x, y, w, h = cv2.boundingRect(np.array(pred_lines))
            bbox = Bbox(x, y, w, h)
            text_bboxes.append(bbox)
        
            contours, _ = cv2.findContours(
                pred_lines, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
            )

            """for contour in contours:
                min_rect = cv2.minAreaRect(contour[0])
                box = cv2.boxPoints(min_rect)
                box = np.intp(box)
                lines.append(box)
            """
            filtered_contours = []
            for contour in contours:
                area = cv2.contourArea(contour)

                if area > area_threshold:
                    filtered_contours.append(contour)


            for _, contour in enumerate(filtered_contours):
                x, y, w, h = cv2.boundingRect(contour)
                x_center = x + (w // 2)
                y_center = y + (h // 2)
                
                line_data = LineData(
                    contour=contour,
                    bbox=Bbox(x, y, w, h),
                    center=(x_center, y_center))
                
                lines.append(line_data)
            
            prediction_images["lines"] = pred_lines
        else:
            prediction_images["lines"] = None


        if np.sum(pred_images) > 0:
            contours, _ = cv2.findContours(
                pred_images, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
            )
            
            for contour in contours:
                x, y, w, h = cv2.boundingRect(contour)
                bbox = Bbox(x, y, w, h)
                images.append(bbox)

            prediction_images["images"] = pred_lines
        else:
            prediction_images["images"] = None


        if np.sum(pred_margin) > 0:
            contours, _ = cv2.findContours(
                pred_margin, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
            )
            
            for contour in contours:
                x, y, w, h = cv2.boundingRect(contour)
                bbox = Bbox(x, y, w, h)
                margins.append(bbox)

            prediction_images["margins"] = pred_margin
        else:
            prediction_images["margins"] = None


        if np.sum(pred_caption) > 0:
            contours, _ = cv2.findContours(
                pred_caption, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
            )

            for contour in contours:
                x, y, w, h = cv2.boundingRect(contour)
                bbox = Bbox(x, y, w, h)
                captions.append(bbox)

            prediction_images["captions"] = pred_caption
        else:
            prediction_images["captions"] = None
            

        layout_data = LayoutData(
            images, 
            text_bboxes, 
            lines, 
            captions, 
            margins,            
            predictions=prediction_images)

        return layout_data


def get_line_treshold(line_prediction: np.array, slice_width: int = 20, steps: int = 8):
    """
    This function generates n slices (of n = steps) width the width of slice_width across the bbox of the detected lines.
    The slice with the max. number of contained contours is taken to be the canditate to calculate the bbox center of each contour and
    take the median distance between each bbox center as estimated line cut-off threshold to sort each line segment across the horizontal

    Note: This approach might turn out to be problematic in case of sparsely spread line segments across a page
    """
    
    slice_width = 20
    steps = 8
    x, y, w, h = cv2.boundingRect(line_prediction)
    bbox_width = w - x
    x_steps = (bbox_width // steps) - 1 - slice_width

    bbox_numbers = []

    for step in range(1, steps+1):
        x_offset = x_steps * step
        x_start = x+x_offset
        x_end = x_start+slice_width
        #print(f"{x_start}:{x_end}")

        slice = line_prediction[y:y+h, x_start:x_end]
        contours, _ = cv2.findContours(
                    slice, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
                )
        #bbox = cv2.boundingRect(contour)
        bbox_numbers.append((len(contours), contours))

    sorted_list = sorted(bbox_numbers, key = lambda x : x[0], reverse=True)
    reference_slice = sorted_list[0]

    y_points = []
    n_contours, contours = reference_slice

    for _, contour in enumerate(contours):
        x, y, w, h  = cv2.boundingRect(contour)
        #x_center = x + (w // 2)
        y_center = y + (h // 2)
        y_points.append(y_center)
        
    
    line_threshold = np.median(y_points) // n_contours
    #print(f"line_treshold: {line_threshold}")

    return line_threshold

def get_line_bbox_centers(layout_data: LayoutData) -> list[tuple[int, int]]:
    centers = []

    for line in layout_data.lines:
        x_center = line.bbox.x + (line.bbox.w // 2)
        y_center = line.bbox.y + (line.bbox.h // 2)
       
        centers.append((x_center, y_center))

    return centers


def group_bbox_centers(bbox_centers: list[tuple[int, int]], line_threshold: int = 20):
  
  sorted_bbox_centers = []
  tmp_line = []

  for i in range(0, len(bbox_centers)):    
      if len(tmp_line) > 0:
        for s in range(0, len(tmp_line)):
          y_diff = abs(tmp_line[s][1] - bbox_centers[i][1])

          if y_diff > line_threshold:
            tmp_line.sort(key=lambda x:x[0])
            sorted_bbox_centers.append(tmp_line.copy())
            tmp_line.clear()

            tmp_line.append(bbox_centers[i])
            break
          else:
            tmp_line.append(bbox_centers[i])
            break
      else:
        tmp_line.append(bbox_centers[i])

  sorted_bbox_centers.append(tmp_line)

  for y in sorted_bbox_centers:
    y.sort(key=lambda x:x[0])

  sorted_bbox_centers = list(reversed(sorted_bbox_centers))

  return sorted_bbox_centers


def group_line_chunks(sorted_bbox_centers, layout_data: LayoutData, debug: bool = False):
    new_line_data = []
    for bbox_centers in sorted_bbox_centers:

      if len(bbox_centers) > 1: # i.e. more than 1 bbox center in a group
          contour_stack = []
          
          for box_center in bbox_centers:
            for line_data in layout_data.lines:
              if box_center == line_data.center:
                  #print("Found yah! Chunk ;)")
                  contour_stack.append(line_data.contour)
                  break
        
          stacked_contour = np.vstack(contour_stack)
          stacked_contour = cv2.convexHull(stacked_contour)
          x, y, w, h = cv2.boundingRect(stacked_contour)
          _bbox = Bbox(x, y, w, h)
          x_center = _bbox.x + (_bbox.w // 2)
          y_center = _bbox.y + (_bbox.h // 2)

          new_line = LineData(
             contour=stacked_contour,
             bbox=_bbox,
             center = (x_center, y_center))
          
          new_line_data.append(new_line)
         

      else:
        for _bcenter in bbox_centers:
            for line_data in layout_data.lines:
              if _bcenter == line_data.center:      
                new_line_data.append(line_data)
                break
        """
        for line_data in layout_data.lines:
              if line_data.center == line_data.center:
                 print(line_data.center)
                 new_line_data.append(line_data)
                 break
        """       
    return new_line_data

### Run Layout inference

In [9]:
data_path = Path("D:/Data/LayoutAnalysis/AllWorks")

for data_subdir in data_path.iterdir():

    if "_add" in data_subdir.name:
        print(data_subdir.name)

        sub_path = Path(data_subdir)

        for sub_sub_dir in sub_path.iterdir():
            images = natsorted(glob(f"{sub_sub_dir}/*.jpg"))
            xml_out = os.path.join(sub_sub_dir, "xml")
            xml_page_out = os.path.join(xml_out, "page")

            if not os.path.exists(xml_out):
                create_dir(xml_out)
                create_dir(xml_page_out)

                for idx, image in tqdm(enumerate(images), total=len(images)):
                    image_name = os.path.basename(images[idx]).split(".")[0]
                    _img = cv2.imread(images[idx])

                    if _img is not None:
                        p_img = resize_to_width(_img, target_width=2048)
                        p_img = cv2.cvtColor(p_img, cv2.COLOR_BGR2RGB)
                        p_img, angle = rotate_from_hough(p_img)

                        _prediction, class_image = predict_image(p_img)
                        _prediction = cv2.resize(_prediction, (_img.shape[1], _img.shape[0]))
                        class_image = cv2.resize(class_image, (_img.shape[1], _img.shape[0]))
                        p_img = cv2.resize(p_img, (_img.shape[1], _img.shape[0]))

                        _prediction = _prediction.astype(np.uint8)

                        
                        layout_data = generate_page_data(_prediction, area_threshold=5)
                        preview_img = generate_preview(p_img, layout_data)
                        
                        if len(layout_data.lines) > 0:
                            bbox_centers = get_line_bbox_centers(layout_data)
                            line_treshold = get_line_treshold(layout_data.predictions["lines"])
                            sorted_bbox_centers = group_bbox_centers(bbox_centers, line_threshold=line_treshold)
                            new_lines = group_line_chunks(sorted_bbox_centers, layout_data)
                            layout_data.lines = new_lines

                            img_out = f"{xml_out}/{image_name}.jpg"
                            p_img = cv2.cvtColor(p_img, cv2.COLOR_BGR2RGB)
                            cv2.imwrite(img_out, p_img)
                        
                            xml_exporter = PageXMLExporter(output_dir=xml_page_out)
                            xml_exporter.export(p_img, image_name, layout_data, [])  

W00EGS1016299_add


  0%|          | 0/926 [00:00<?, ?it/s]d:\Projects\OCR-Inference\venv\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Projects\OCR-Inference\venv\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 926/926 [27:42<00:00,  1.80s/it]


W00KG02390_add


100%|██████████| 202/202 [00:58<00:00,  3.44it/s]


W0TTBBC0006_add


100%|██████████| 465/465 [03:04<00:00,  2.52it/s]


... (22 outputs left out)

W4CZ7445_was_in_training_data_add_more_maybe


100%|██████████| 100/100 [00:48<00:00,  2.06it/s]
0it [00:00, ?it/s]


W4PD1180_add


100%|██████████| 100/100 [00:34<00:00,  2.89it/s]
